# 3.3. 기본 CF 알고리즘

$$
\hat{r}(u, m)
= \frac{\sum_{v \in U_m} s(u, v)\, r(v, m)}
       {\sum_{v \in U_m} s(u, v)}
$$

- $u$ : 예측 대상 사용자  
- $m$ : 예측 대상 영화  
- $U_m$ : 영화 $m$을 평가한 사용자 집합  
- $s(u, v)$ : 사용자 $u$와 $v$ 간의 유사도  
- $r(v, m)$ : 사용자 $v$가 영화 $m$에 부여한 평점  
- $\hat{r}(u, m)$ : 사용자 $u$의 영화 $m$에 대한 예측 평점


코드와 수식 대응 관계:

$$
\texttt{np.dot(sim\_scores, movie\_ratings)}
= \sum_{v \in U_m} s(u, v)\, r(v, m)
$$

$$
\texttt{sim\_scores.sum()}
= \sum_{v \in U_m} s(u, v)
$$

사용자 $u$의 영화 $m$에 대한 예측 평점은,  
영화 $m$을 평가한 사용자들의 평점을 유사도로 가중 평균한 값이다.


In [ ]:
# 데이터 읽어오기(user, item, data)
import os
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

base_src = Path.cwd().parent / 'data'

# user 데이터
u_user_src = os.path.join(base_src,'u.user')
u_cols = ['user_id','age','sex','occupation','zip_code']
users = pd.read_csv(u_user_src,
	sep = '|',
        names = u_cols,
        encoding = 'latin-1')
users: pd.DataFrame = users.set_index('user_id')

# movie 데이터
u_item_src = os.path.join(base_src,'u.item')
i_cols = ['movie_id','title','release date','video release date','IMDB URL','unknown','Action','Adventure','Animation','Children\'s','Comedy','Crime','Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western']
movies = pd.read_csv(u_item_src,
	    sep='|',
            names=i_cols,
            encoding='latin-1')
movies: pd.DataFrame = movies[['movie_id','title']]

# rating 데이터
u_data_src = os.path.join(base_src,'u.data')
r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(u_data_src,
        sep = '\t',
        names = r_cols,
        encoding='latin-1')
ratings: pd.DataFrame = ratings.drop('timestamp',axis=1)

# RMSE 함수
def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

# score(RMSE) 계산하는 함수
def score(model):
    id_pairs = zip(x_test['user_id'], x_test['movie_id'])
    y_pred = np.array([model(user, movie) for (user, movie) in id_pairs])
    y_true = np.array(x_test['rating'])
    return RMSE(y_true, y_pred)

# 데이터 셋
x = ratings.copy()
y = ratings['user_id']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y)
ratings_matrix = x_train.pivot(index='user_id',columns = 'movie_id',values='rating')


In [35]:
# 코사인 유사도 계산

from sklearn.metrics.pairwise import cosine_similarity

matrix_copy = ratings_matrix.copy().fillna(0)

user_similarity = cosine_similarity(matrix_copy, matrix_copy)
user_similarity = pd.DataFrame(user_similarity, index=ratings_matrix.index, columns=ratings_matrix.index)

# 주어진 영화의 가중평균 rating을 계산하는 함수
def CF_simple(user_id, movie_id):
    if movie_id in ratings_matrix.columns:
        sim_scores = user_similarity[user_id].copy()
        movie_ratings = ratings_matrix[movie_id].copy()
        none_rating_idx = movie_ratings[movie_ratings.isnull()].index # 영화를 평가하지 않은 사용자 아이디 추출
        movie_ratings = movie_ratings.drop(none_rating_idx)
        sim_scores = sim_scores.drop(none_rating_idx)
        return np.dot(sim_scores, movie_ratings) / sim_scores.sum()
    else:
        return 3.0

score(CF_simple)


np.float64(1.0241654866729961)

# 3.4. 이웃을 고려한 CF

모든 사용자에 대한 가중 평균이 아닌, 정말 유사한 사용자에 대한 가중 평균을 구하는 방식. 

**"정말 유사한 사용자"의 기준?**

1. K Nearest Neighbors (KNN) 방법

- 미리 K명을 정하고, 유사도가 높은 순 K명을 뽑아서 사용하는 방법

2. Thresholding 방법

- K를 미리 정하지 않고, 상관 계수 또는 코사인 계수가 threshold 이상인 사용자를 유사 집단으로 정의하는 방법

일반적으로 Thresholding 방법이 KNN 방법보다 정확도가 높긴 하지만, threshold 이상인 사용자를 찾지 못할 경우 추천이 어려워짐. 
그래서 이따금씩 KNN 방법도 사용됨. 


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

###### 데이터 불러오기 및 데이터 셋 만들기 ######
base_src = Path.cwd().parent / 'data'
u_user_src = os.path.join(base_src,'u.user')
u_cols = ['user_id','age','sex','occupation','zip_code']
users = pd.read_csv(u_user_src,
	sep = '|',
        names = u_cols,
        encoding = 'latin-1')
users = users.set_index('user_id')

u_item_src = os.path.join(base_src,'u.item')
i_cols = ['movie_id','title','release date','video release date','IMDB URL','unknown','Action','Adventure','Animation','Children\'s','Comedy','Crime','Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western']
movies = pd.read_csv(u_item_src,
	    sep='|',
            names=i_cols,
            encoding='latin-1')
movies = movies.set_index('movie_id')

u_data_src = os.path.join(base_src,'u.data')
r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(u_data_src,
        sep = '\t',
        names = r_cols,
        encoding='latin-1')

def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

##################################################################################
# 유사집단의 크기를 미리 정하기 위해서 기존 score 함수에 neighbor_size 인자값 추가
def score(model,neighbor_size=0):
    # 테스트 데이터의 user_id와 movie_id 간 pair를 맞춰 튜플형원소 리스트데이터를 만듬
    id_pairs = zip(x_test['user_id'],x_test['movie_id'])
    # 모든 사용자-영화 짝에 대해서 주어진 예측모델에 의해 예측값 계산 및 리스트형 데이터 생성
    y_pred = np.array([model(user,movie,neighbor_size) for (user,movie) in id_pairs])
    # 실제 평점값
    y_true = np.array(x_test['rating'])
    return RMSE(y_true, y_pred)

##################################################################################
x = ratings.copy()
y = ratings['user_id']

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.25,stratify=y)

rating_matrix = x_train.pivot(index='user_id',columns = 'movie_id',values='rating')

###### train set의 모든 가능한 사용자 pair의 cosine similarities 계산 ######
# 코사인 유사도를 계산하는 사이킷런의 라이브러리
from sklearn.metrics.pairwise import cosine_similarity
# 코사인 유사도를 구하기 위해 rating값을 복사하고, 계산 시 NaN값 에러 대비를 위해 결측치를 0으로 대체
matrix_dummy = rating_matrix.copy().fillna(0)
# 모든 사용자간 코사인 유사도 구함
user_similarity = cosine_similarity(matrix_dummy,matrix_dummy)
# 필요한 값 조회를 위해 인덱스 및 컬럼명 지정
user_similarity = pd.DataFrame(user_similarity,
                     index = rating_matrix.index,
                     columns = rating_matrix.index)

##################################################################################
###### Neighbor size를 정해서 예측치를 계산하는 함수 ######
def CF_knn(user_id,movie_id,neighbor_size=0):
  if movie_id in rating_matrix.columns:
    sim_scores = user_similarity[user_id].copy()
    movie_ratings = rating_matrix[movie_id].copy()
    none_rating_idx = movie_ratings[movie_ratings.isnull()].index
    movie_ratings = movie_ratings.dropna()
    sim_scores = sim_scores.drop(none_rating_idx)

    if neighbor_size == 0:
      return np.dot(sim_scores,movie_ratings) / sim_scores.sum()

    else:
      if len(sim_scores) > 1:
        neighbor_size = min(neighbor_size,len(sim_scores))
        sim_scores = np.array(sim_scores)
        movie_ratings = np.array(movie_ratings)
        user_idx = np.argsort(sim_scores) # 유사도가 낮은 순서대로 정렬된 인덱스 반환
        sim_scores = sim_scores[user_idx][-neighbor_size:] # 유사도가 낮은 순서대로 정렬된 인덱스 중 뒤에서부터 neighbor_size개만 추출
        movie_ratings = movie_ratings[user_idx][-neighbor_size:]
        return np.dot(sim_scores,movie_ratings) / sim_scores.sum()
      else:
        return 3.0
  return 3.0


# 정확도 계산
score(CF_knn,neighbor_size=30)

np.float64(1.0125530362442066)

In [43]:
##### 실제 주어진 사용자에 대해 추천을 받는 기능 구현 #####
rating_matrix = ratings.pivot_table(values='rating',
                                    index='user_id',
                                    columns='movie_id')
matrix_dummy = rating_matrix.copy().fillna(0)
user_similarity = cosine_similarity(matrix_dummy,matrix_dummy)
user_similarity = pd.DataFrame(user_similarity,
                               index=rating_matrix.index,
                               columns=rating_matrix.index)

def recom_movie(user_id,n_items,neighbor_size=30):
  user_movie = rating_matrix.loc[user_id].copy()

  for movie in rating_matrix.columns:
    if pd.notnull(user_movie.loc[movie]): # 사용자가 이미 평가한 영화는 제외
      user_movie.loc[movie] = 0

    else:
      user_movie.loc[movie] = CF_knn(user_id,movie,neighbor_size) # 사용자가 평가하지 않은 영화는 CF_knn 함수를 통해 예측 평점 계산

  movie_sort = user_movie.sort_values(ascending=False)[:n_items] # 예측 평점이 높은 순서대로 정렬
  recom_movies = movies.loc[movie_sort.index] # 영화 제목 추출
  recommendations = recom_movies['title'] # 영화 제목 리스트 반환
  return recommendations

recom_movie(user_id=729,n_items=5,neighbor_size=30)



movie_id
1293                         Star Kid (1997)
1467    Saint of Fort Washington, The (1993)
1189                      Prefontaine (1997)
1500               Santa with Muscles (1996)
22                         Braveheart (1995)
Name: title, dtype: object

# 3.5. 최적의 이웃 크기 결정

neighbor_size를 적절하게 결정하는 것이 중요. 

너무 크게 잡게 되면, 사실상 best_seller 방식과 크게 차이가 없게 되버림.
너무 작게 잡게 되면, 추천 데이터의 신뢰도가 낮아짐. (극단적으로 생각해보면, 나랑 A 영화를 똑같이 본 한 사람만 고려하는 셈)

이러한 현상을 ML에서는 오버피팅이라고 일컬음. 이 현상을 해결하기 위해서는 여러 실험을 해보면서 최적의 이웃의 크기를 찾아내야 함.

![최적의 이웃 크기](../static/img_1.png)

In [ ]:
# neighbor size가 10,20,30,40,50,60인 경우에 대해서 RMSE를 계산하고 이를 출력한다.
for neighbor_size in [10,20,30,40,50,60]:
  print('Neighbor size = %d : RMSE = %.4f'%(neighbor_size,score(CF_knn,neighbor_size)))

Neighbor size = 10 : RMSE = 0.8093
Neighbor size = 20 : RMSE = 0.8776
Neighbor size = 30 : RMSE = 0.9038
Neighbor size = 40 : RMSE = 0.9186
Neighbor size = 50 : RMSE = 0.9279
Neighbor size = 60 : RMSE = 0.9343
